# ДЗ-1: аналитическая модель стоимости CNN

Ноутбук делает всё от начала до конца: пишет код, меряет сеть на GPU, калибрует latency и energy,
рисует графики и упаковывает `results/` в архив.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**. Потом Runtime → **Run all**.
Замеры идут примерно 10–15 минут.

In [ ]:
!nvidia-smi
!pip -q install nvidia-ml-py

In [ ]:
%%writefile models.py
"""Сеть из задания ДЗ-1 (последовательная CNN, без residual).

Слой                                   Выход
Conv7x7 s2 3->32 + ReLU, MaxPool 3x3 s2 p1   32 x S/4 x S/4
Conv5x5 32->64 + ReLU                      64 x S/4
Conv3x3 s2 64->128 + ReLU                 128 x S/8
Conv1x1 128->256 + ReLU                   256 x S/8
Conv3x3 s2 256->256 + ReLU                256 x S/16
Conv1x1 256->512 + ReLU                   512 x S/16
GlobalAvgPool, Linear 512->256, ReLU, Linear 256->100

Все свёртки: padding = k // 2, bias=False. Все ReLU: inplace=True.
BatchNorm в таблице модели нет, поэтому его нет и здесь.

nn.Sequential выбран специально: его forward — это цикл
`for m in self: x = m(x)`, поэтому порядок жизни тензоров известен точно
и пиковую память можно посчитать по формуле (см. equations.py).
"""
import torch
from torch import nn


def conv(cin, cout, k, s=1):
    return nn.Conv2d(cin, cout, kernel_size=k, stride=s, padding=k // 2, bias=False)


def build_model(num_classes: int = 100) -> nn.Sequential:
    return nn.Sequential(
        conv(3, 32, 7, 2), nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        conv(32, 64, 5), nn.ReLU(inplace=True),
        conv(64, 128, 3, 2), nn.ReLU(inplace=True),
        conv(128, 256, 1), nn.ReLU(inplace=True),
        conv(256, 256, 3, 2), nn.ReLU(inplace=True),
        conv(256, 512, 1), nn.ReLU(inplace=True),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        nn.Linear(512, 256), nn.ReLU(inplace=True),
        nn.Linear(256, num_classes),
    )


if __name__ == "__main__":
    m = build_model()
    print(m)
    print("параметров:", sum(p.numel() for p in m.parameters()))
    with torch.inference_mode():
        for S in (32, 224, 512):
            print(S, tuple(m(torch.randn(2, 3, S, S)).shape))

In [ ]:
%%writefile equations.py
"""Аналитическая модель стоимости сети из models.py.

Все функции принимают числа или numpy-массивы S (image_size) и B (batch)
и работают через broadcasting, поэтому поверхность над плоскостью (S, B)
строится одним вызовом.

Обозначения: S кратно 16, все тензоры FP32 (4 байта).
Размеры карт признаков: после conv1 S/2, после maxpool S/4, дальше S/4, S/8, S/8, S/16, S/16.
"""
import numpy as np

FP32 = 4
INT64 = 8
NUM_CLASSES = 100

# --------------------------------------------------------------- параметры
# (имя, число параметров)
PARAMS = [
    ("conv1.w", 32 * 3 * 7 * 7),        # 4 704
    ("conv2.w", 64 * 32 * 5 * 5),       # 51 200
    ("conv3.w", 128 * 64 * 3 * 3),      # 73 728
    ("conv4.w", 256 * 128 * 1 * 1),     # 32 768
    ("conv5.w", 256 * 256 * 3 * 3),     # 589 824
    ("conv6.w", 512 * 256 * 1 * 1),     # 131 072
    ("fc1.w", 256 * 512), ("fc1.b", 256),
    ("fc2.w", NUM_CLASSES * 256), ("fc2.b", NUM_CLASSES),
]
N_PARAMS = sum(n for _, n in PARAMS)    # 1 040 324


def alloc(nbytes):
    """Размер блока, который выдаёт caching allocator PyTorch:
    запрос округляется вверх до кратного 512 байт (пустой тензор не занимает памяти)."""
    nbytes = np.asarray(nbytes, dtype=np.float64)
    return np.where(nbytes > 0, 512.0 * np.ceil(nbytes / 512.0), 0.0)


WEIGHT_BYTES = float(sum(alloc(n * FP32) for _, n in PARAMS))

# Постоянная добавка библиотек: рабочая область cuBLAS, которую PyTorch
# выделяет через тот же allocator при первом матричном умножении и больше
# не освобождает. Не зависит от S и B. Значение по умолчанию — дефолт PyTorch
# для GPU до Hopper (CUBLAS_WORKSPACE_CONFIG=:4096:2:16:8):
# 2 * 4096 KiB + 8 * 16 KiB. measure.py измеряет её напрямую (не подбирает
# по целевой метрике) и сохраняет в results/theta.json.
RUNTIME_BYTES_DEFAULT = 2 * 4096 * 1024 + 8 * 16 * 1024


# ------------------------------------------------- размеры активаций (на 1 картинку)
def act_elems(S):
    """Число элементов выходного тензора каждого слоя для одной картинки."""
    S = np.asarray(S, dtype=np.float64)
    return {
        "x":   3 * S ** 2,                 # вход
        "a1": 32 * (S / 2) ** 2,           # conv1        = 8 S^2
        "a2": 32 * (S / 4) ** 2,           # maxpool      = 2 S^2
        "idx": 32 * (S / 4) ** 2,          # индексы maxpool (int64) = 2 S^2
        "a3": 64 * (S / 4) ** 2,           # conv2        = 4 S^2
        "a4": 128 * (S / 8) ** 2,          # conv3        = 2 S^2
        "a5": 256 * (S / 8) ** 2,          # conv4        = 4 S^2
        "a6": 256 * (S / 16) ** 2,         # conv5        = 1 S^2
        "a7": 512 * (S / 16) ** 2,         # conv6        = 2 S^2
        "g": 512 + 0 * S,                  # global avg pool
        "f1": 256 + 0 * S,                 # fc1
        "f2": NUM_CLASSES + 0 * S,         # fc2
    }


# --------------------------------------------------------------------- FLOPs
def layer_table(S, B):
    """FLOPs и минимальный трафик DRAM (байты) каждого ядра (kernel) прямого прохода.

    FLOPs: 1 MAC = 2 FLOPs. Для свёртки 2 * B * Cout * Hout * Wout * Cin * k^2.
    Трафик: каждое ядро один раз читает свои входы (включая веса) и один раз
    пишет выход. ReLU inplace — отдельное ядро: читает и пишет тензор.
    Поэлементные операции (ReLU, сравнения в maxpool, сложения в GAP, bias)
    дают ~0,2% FLOPs; они учтены отдельно в поле aux_flops.
    """
    S = np.asarray(S, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    e = act_elems(S)
    rows = []

    def add(name, flops, read_elems, write_bytes, weight_params=0, aux=0.0):
        rows.append({
            "name": name,
            "flops": flops,
            "bytes": FP32 * B * read_elems + write_bytes + FP32 * weight_params,
            "aux_flops": aux,
        })

    def conv(name, cin_key, out_key, cin, cout, k, so):
        mac = B * cout * so ** 2 * cin * k * k
        add(name, 2 * mac, e[cin_key], FP32 * B * e[out_key], cout * cin * k * k)
        add(name + ".relu", 0 * S, e[out_key], FP32 * B * e[out_key], aux=B * e[out_key])

    conv("conv1", "x", "a1", 3, 32, 7, S / 2)
    add("maxpool", 0 * S, e["a1"], FP32 * B * e["a2"] + INT64 * B * e["idx"], aux=8 * B * e["a2"])
    conv("conv2", "a2", "a3", 32, 64, 5, S / 4)
    conv("conv3", "a3", "a4", 64, 128, 3, S / 8)
    conv("conv4", "a4", "a5", 128, 256, 1, S / 8)
    conv("conv5", "a5", "a6", 256, 256, 3, S / 16)
    conv("conv6", "a6", "a7", 256, 512, 1, S / 16)
    add("gap", 0 * S, e["a7"], FP32 * B * e["g"], aux=B * e["a7"])
    add("fc1", 2 * B * 512 * 256 + 0 * S, e["g"], FP32 * B * e["f1"], 512 * 256 + 256, aux=B * 256)
    add("fc1.relu", 0 * S, e["f1"], FP32 * B * e["f1"], aux=B * 256)
    add("fc2", 2 * B * 256 * NUM_CLASSES + 0 * S, e["f1"], FP32 * B * e["f2"], 256 * NUM_CLASSES + NUM_CLASSES, aux=B * NUM_CLASSES)
    return rows


N_KERNELS = 17   # 6 conv + 6 relu + maxpool + gap + fc1 + relu + fc2


def flops(image_size, batch):
    """FLOPs одного прямого прохода (свёртки и линейные слои, 1 MAC = 2 FLOPs).

    Закрытая форма: FLOPs = 17 712 * B * S^2 + 313 344 * B.
    """
    return sum(r["flops"] for r in layer_table(image_size, batch))


def aux_flops(image_size, batch):
    """Поэлементные операции (ReLU, maxpool, GAP, bias): ~39 * B * S^2, около 0,2% от flops()."""
    return sum(r["aux_flops"] for r in layer_table(image_size, batch))


def bytes_moved(image_size, batch):
    """Минимальный трафик DRAM одного прямого прохода, байты.

    Закрытая форма: 380 * B * S^2 + 4 161 296 (веса) + 8 592 * B (голова).
    """
    return sum(r["bytes"] for r in layer_table(image_size, batch))


# -------------------------------------------------------------------- память
def activation_timeline(image_size, batch):
    """Сколько байт активаций живёт в момент выполнения каждого слоя.

    nn.Sequential: во время слоя живы его вход и его выход; предыдущий
    тензор освобождается, когда цикл перезаписывает переменную.
    Вход x держит вызывающий код, поэтому он учтён отдельно в memory().
    ReLU inplace ничего не выделяет. MaxPool на CUDA вызывает
    max_pool2d_with_indices и временно держит индексы int64.
    """
    S = np.asarray(image_size, dtype=np.float64)
    B = np.asarray(batch, dtype=np.float64)
    e = act_elems(S)
    a = {k: alloc(FP32 * B * v) for k, v in e.items()}
    a["idx"] = alloc(INT64 * B * e["idx"])
    return {
        "conv1":   a["a1"],
        "maxpool": a["a1"] + a["a2"] + a["idx"],
        "conv2":   a["a2"] + a["a3"],
        "conv3":   a["a3"] + a["a4"],
        "conv4":   a["a4"] + a["a5"],
        "conv5":   a["a5"] + a["a6"],
        "conv6":   a["a6"] + a["a7"],
        "gap":     a["a7"] + a["g"],
        "fc1":     a["g"] + a["f1"],
        "fc2":     a["f1"] + a["f2"],
    }


def memory(image_size, batch, runtime_bytes=RUNTIME_BYTES_DEFAULT):
    """Пик torch.cuda.max_memory_allocated() за один прямой проход, байты.

    = веса + вход + рабочая область cuBLAS + максимум по слоям живых активаций.
    Максимум всегда на maxpool: a1 + a2 + индексы = (32 + 8 + 16) * B * S^2 байт.
    Закрытая форма (без округления до 512 Б):
        Memory = 4 161 536 + runtime_bytes + 12 * B * S^2 + 56 * B * S^2
               = 4 161 536 + runtime_bytes + 68 * B * S^2.
    Здесь ничего не подбирается: формула считает память точно, с точностью
    до округления allocator-а (512 Б). Не учтена только рабочая область cuDNN
    внутри свёрток: её размер выбирает эвристика cuDNN, measure.py показывает,
    есть ли она.
    """
    S = np.asarray(image_size, dtype=np.float64)
    B = np.asarray(batch, dtype=np.float64)
    x = alloc(FP32 * B * 3 * S ** 2)
    peak_act = np.max(np.stack(np.broadcast_arrays(*activation_timeline(S, B).values())), axis=0)
    return WEIGHT_BYTES + x + runtime_bytes + peak_act


# ------------------------------------------------------------------- latency
def layer_times(image_size, batch, theta):
    """Время каждого ядра по roofline: max(FLOPs / P, bytes / BW)."""
    P, BW = theta["peak_flops"], theta["bandwidth"]
    return [np.maximum(r["flops"] / P, r["bytes"] / BW) for r in layer_table(image_size, batch)]


def latency(image_size, batch, theta):
    """Время одного прямого прохода, секунды.

    Latency = t0 + sum_k max(FLOPs_k / P, Bytes_k / BW)
    theta = {t0: постоянная задержка (запуски 17 ядер, Python, синхронизация), с;
             peak_flops: достижимая производительность свёрток, FLOP/s;
             bandwidth: достижимая пропускная способность памяти, байт/с}
    Режимы: t0 доминирует — launch-bound; доминируют ядра, где bytes/BW > FLOPs/P —
    memory-bound; доминируют свёртки с FLOPs/P > bytes/BW — compute-bound.
    """
    return theta["t0"] + sum(layer_times(image_size, batch, theta))


def regime(image_size, batch, theta):
    """0 = launch-bound, 1 = memory-bound, 2 = compute-bound: какая часть времени больше."""
    t_mem = 0.0
    t_cmp = 0.0
    for r in layer_table(image_size, batch):
        tc, tm = r["flops"] / theta["peak_flops"], r["bytes"] / theta["bandwidth"]
        t_cmp = t_cmp + np.where(tc >= tm, tc, 0.0)
        t_mem = t_mem + np.where(tc < tm, tm, 0.0)
    t0 = theta["t0"] + 0 * t_mem
    return np.argmax(np.stack(np.broadcast_arrays(t0, t_mem, t_cmp)), axis=0)


# -------------------------------------------------------------------- energy
def energy(image_size, batch, theta_energy, theta=None):
    """Энергия одного прямого прохода всей GPU, джоули.

    Energy = P_static * Latency(S, B, theta) + e_flop * FLOPs + e_byte * Bytes
    theta_energy = {p_static: мощность, которую GPU тратит, пока занята (Вт);
                    e_flop: Дж на FLOP; e_byte: Дж на байт DRAM-трафика}
    Если theta (параметры latency) не передан, он берётся из theta_energy["latency_theta"].
    """
    th = theta if theta is not None else theta_energy["latency_theta"]
    return (theta_energy["p_static"] * latency(image_size, batch, th)
            + theta_energy["e_flop"] * flops(image_size, batch)
            + theta_energy["e_byte"] * bytes_moved(image_size, batch))


if __name__ == "__main__":
    print("параметров:", N_PARAMS, " веса, байт:", WEIGHT_BYTES)
    for S, B in [(32, 1), (224, 64), (512, 256)]:
        print(f"S={S:3d} B={B:3d}  FLOPs={flops(S, B):.4g}  bytes={bytes_moved(S, B):.4g}  "
              f"memory={memory(S, B) / 2**20:.1f} MiB")

In [ ]:
%%writefile measure.py
"""Замеры на GPU: latency, пиковая память, энергия по сетке (S, B).

Запуск (Colab / Kaggle, GPU):  python measure.py
Результаты: results/measurements.csv, results/runtime.json,
            results/flops_check.csv, results/memory_audit.csv
"""
import csv
import json
import math
import os
import platform
import time
from pathlib import Path

import numpy as np
import torch

import equations as eq
from models import build_model

RES = Path(__file__).resolve().parent / "results"
SEED = 2026

BASE_S = [32, 64, 128, 224, 256, 384, 512]
BASE_B = [1, 2, 4, 8, 16, 32, 64, 128, 256]

LAT_MIN_REPS, LAT_MIN_SEC, LAT_MAX_REPS = 10, 0.3, 200
ENERGY_MIN_SEC, ENERGY_MIN_REPS = 2.0, 3


def make_grid(seed=SEED):
    """База + случайные точки. Случайные S: 4 кратных 16 из [32, 512] вне базы;
    случайные B: 3 целых из [1, 256], не степени двойки. Конфигурация попадает
    в валидацию, если её S или B случайные: калибруем только на базовой сетке
    7 x 9 = 63 точки, проверяем на остальных 69 (их модель не видела)."""
    rng = np.random.default_rng(seed)
    s_pool = [s for s in range(32, 513, 16) if s not in BASE_S]
    b_pool = [b for b in range(1, 257) if b & (b - 1) != 0]
    rand_s = sorted(int(s) for s in rng.choice(s_pool, 4, replace=False))
    rand_b = sorted(int(b) for b in rng.choice(b_pool, 3, replace=False))
    grid = []
    for S in sorted(BASE_S + rand_s):
        for B in sorted(BASE_B + rand_b):
            grid.append((S, B, (S in rand_s) or (B in rand_b)))
    return grid, rand_s, rand_b


class Energy:
    """Счётчик энергии всей GPU через NVML (миллиджоули с момента загрузки драйвера)."""

    def __init__(self):
        self.h = None
        try:
            import pynvml
            pynvml.nvmlInit()
            self.nv = pynvml
            self.h = pynvml.nvmlDeviceGetHandleByIndex(torch.cuda.current_device())
            pynvml.nvmlDeviceGetTotalEnergyConsumption(self.h)
        except Exception as e:   # нет pynvml или GPU не поддерживает счётчик
            print("NVML energy counter недоступен:", e)
            self.h = None

    def joules(self):
        return self.nv.nvmlDeviceGetTotalEnergyConsumption(self.h) / 1000.0

    def info(self):
        if self.h is None:
            return {}
        try:
            return {"temp_c": self.nv.nvmlDeviceGetTemperature(self.h, 0),
                    "sm_clock_mhz": self.nv.nvmlDeviceGetClockInfo(self.h, 1),
                    "power_w_now": self.nv.nvmlDeviceGetPowerUsage(self.h) / 1000.0}
        except Exception:
            return {}


def forward(model, x):
    with torch.inference_mode():
        return model(x)


def measure_runtime_bytes(model, dev):
    """Сколько байт allocator держит вне формулы: веса проверяем точно,
    а рабочую область cuBLAS (и всё, что библиотеки выделяют один раз и
    навсегда) измеряем как разницу до и после первого прохода."""
    torch.cuda.synchronize()
    w_alloc = torch.cuda.memory_allocated()
    x = torch.randn(1, 3, 32, 32, device=dev)
    out = forward(model, x)
    torch.cuda.synchronize()
    del out, x
    torch.cuda.synchronize()
    runtime = torch.cuda.memory_allocated() - w_alloc
    return {"weights_allocated": w_alloc, "weights_predicted": eq.WEIGHT_BYTES,
            "runtime_bytes": runtime, "runtime_bytes_default_formula": eq.RUNTIME_BYTES_DEFAULT}


def flops_check(model, dev):
    """Сверка flops() с независимым счётчиком PyTorch (FlopCounterMode, тоже 2 FLOPs на MAC)."""
    from torch.utils.flop_counter import FlopCounterMode
    rows = []
    for S, B in [(32, 1), (64, 3), (224, 8), (512, 2)]:
        x = torch.randn(B, 3, S, S, device=dev)
        fc = FlopCounterMode(display=False)
        with fc:
            forward(model, x)
        rows.append({"S": S, "B": B, "torch_flops": fc.get_total_flops(),
                     "formula_flops": float(eq.flops(S, B))})
        print("FLOPs check", rows[-1])
    return rows


def memory_audit(model, dev, S=224, B=32):
    """Проход слой за слоем: сколько памяти живёт во время каждого слоя.
    Повторяет цикл nn.Sequential, поэтому времена жизни те же, что в forward."""
    torch.cuda.empty_cache()
    x = torch.randn(B, 3, S, S, device=dev)
    torch.cuda.synchronize()
    static = torch.cuda.memory_allocated()          # веса + вход + рабочие области
    pred = eq.activation_timeline(S, B)
    names = {0: "conv1", 2: "maxpool", 3: "conv2", 5: "conv3", 7: "conv4", 9: "conv5",
             11: "conv6", 13: "gap", 15: "fc1", 17: "fc2"}
    rows = []
    with torch.inference_mode():
        h = x
        for i, m in enumerate(model):
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            h = m(h)
            torch.cuda.synchronize()
            peak = torch.cuda.max_memory_allocated() - static
            if i in names:
                rows.append({"S": S, "B": B, "layer": names[i], "measured_bytes": peak,
                             "predicted_bytes": float(pred[names[i]])})
                print("audit", rows[-1])
    del h, x
    return rows


def measure_one(model, dev, S, B, en):
    x = torch.randn(B, 3, S, S, device=dev)
    for _ in range(2):                              # прогрев: выбор алгоритмов cuDNN, кэш
        forward(model, x)
    torch.cuda.synchronize()

    # --- память: чистый allocator, пик за один проход
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    before = torch.cuda.memory_allocated()
    out = forward(model, x)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    req_peak = torch.cuda.memory_stats().get("requested_bytes.all.peak", float("nan"))
    del out

    # --- latency: CUDA events вокруг каждого прохода, медиана
    times = []
    t_start = time.perf_counter()
    while len(times) < LAT_MAX_REPS and (len(times) < LAT_MIN_REPS or time.perf_counter() - t_start < LAT_MIN_SEC):
        a, b = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        a.record()
        forward(model, x)
        b.record()
        torch.cuda.synchronize()
        times.append(a.elapsed_time(b) / 1000.0)
    lat = float(np.median(times))

    # --- энергия: много проходов подряд, разность счётчика NVML / число проходов
    e_j = p_w = float("nan")
    n_e = 0
    if en.h is not None:
        n_e = max(ENERGY_MIN_REPS, math.ceil(ENERGY_MIN_SEC / max(lat, 1e-6)))
        torch.cuda.synchronize()
        e0, t0 = en.joules(), time.perf_counter()
        for _ in range(n_e):
            forward(model, x)
        torch.cuda.synchronize()
        e1, t1 = en.joules(), time.perf_counter()
        e_j = (e1 - e0) / n_e
        p_w = (e1 - e0) / (t1 - t0)
    del x
    return {"status": "ok", "latency_s": lat,
            "latency_p10_s": float(np.percentile(times, 10)), "latency_p90_s": float(np.percentile(times, 90)),
            "n_latency": len(times), "memory_bytes": peak, "memory_before_bytes": before,
            "memory_requested_peak_bytes": req_peak, "energy_j": e_j, "avg_power_w": p_w, "n_energy": n_e}


def main():
    assert torch.cuda.is_available(), "Нужна GPU (Colab: Runtime -> Change runtime type -> T4 GPU)"
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.manual_seed(SEED)
    RES.mkdir(exist_ok=True)
    dev = torch.device("cuda")

    model = build_model().to(dev).eval()
    env = {"gpu": torch.cuda.get_device_name(), "torch": torch.__version__,
           "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
           "python": platform.python_version(),
           "gpu_total_memory_bytes": torch.cuda.get_device_properties(0).total_memory}
    rt = measure_runtime_bytes(model, dev)
    env.update(rt)
    print(env)

    try:
        json.dump(flops_check(model, dev), open(RES / "flops_check.json", "w"), indent=1)
    except Exception as e:      # проверка необязательная, замеры от неё не зависят
        print("FlopCounterMode не сработал:", e)
    audit = memory_audit(model, dev)
    with open(RES / "memory_audit.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(audit[0].keys()))
        w.writeheader()
        w.writerows(audit)

    grid, rand_s, rand_b = make_grid()
    env.update({"random_S": rand_s, "random_B": rand_b, "seed": SEED})
    json.dump(env, open(RES / "runtime.json", "w"), indent=1)
    print("случайные S:", rand_s, " случайные B:", rand_b, " конфигураций:", len(grid))

    en = Energy()
    order = np.random.default_rng(SEED).permutation(len(grid))   # перемешиваем, чтобы нагрев GPU не коррелировал с S и B
    rows = []
    fields = ["S", "B", "is_validation", "status", "latency_s", "latency_p10_s", "latency_p90_s", "n_latency",
              "memory_bytes", "memory_before_bytes", "memory_requested_peak_bytes", "energy_j", "avg_power_w",
              "n_energy", "temp_c", "sm_clock_mhz", "power_w_now"]
    t_all = time.perf_counter()
    for k, i in enumerate(order):
        S, B, val = grid[i]
        row = {"S": S, "B": B, "is_validation": int(val)}
        try:
            row.update(measure_one(model, dev, S, B, en))
        except torch.cuda.OutOfMemoryError:
            row.update({"status": "OOM"})
        torch.cuda.empty_cache()
        row.update(en.info())
        rows.append(row)
        print(f"[{k + 1:3d}/{len(grid)}] S={S:3d} B={B:3d} {row['status']}"
              + (f"  lat={row['latency_s'] * 1e3:8.3f} ms  mem={row['memory_bytes'] / 2**20:8.1f} MiB"
                 f"  E={row['energy_j']:.4f} J" if row["status"] == "ok" else ""), flush=True)
        with open(RES / "measurements.csv", "w", newline="") as f:      # сохраняем после каждой точки
            w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
            w.writeheader()
            for r in sorted(rows, key=lambda r: (r["S"], r["B"])):
                w.writerow(r)
    print(f"готово за {time.perf_counter() - t_all:.0f} с")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile calibrate.py
"""Калибровка theta для latency и energy. Только numpy.

Калибруем ТОЛЬКО на базовой сетке (is_validation = 0), качество считаем
отдельно на обучающих и на валидационных точках (случайные S или B,
которые модель не видела). Память не калибруется: она считается точно.

Запуск: python calibrate.py  ->  results/theta.json, results/predictions.csv
"""
import csv
import json
from pathlib import Path

import numpy as np

import equations as eq

RES = Path(__file__).resolve().parent / "results"


def load(path=RES / "measurements.csv"):
    rows = list(csv.DictReader(open(path)))
    out = {}
    for k in rows[0]:
        col = []
        for r in rows:
            v = r[k]
            try:
                col.append(float(v))
            except ValueError:
                col.append(np.nan if v == "" else v)
        is_text = any(isinstance(v, str) for v in col)
        out[k] = np.array(col, dtype=object if is_text else float)
    return out


# ------------------------------------------------------------------ latency
# Паспортные пределы GPU (FP32 FLOP/s, байт/с DRAM). Нужны как верхние границы для P и BW.
GPU_SPECS = {"T4": (8.14e12, 320e9), "P100": (9.3e12, 732e9), "V100": (15.7e12, 900e9),
             "A100": (19.5e12, 1555e9), "L4": (30.3e12, 300e9), "A10": (31.2e12, 600e9)}


def gpu_bounds(name):
    for k, v in GPU_SPECS.items():
        if k in name:
            return v
    return (np.inf, np.inf)


def fit_latency(S, B, y, p_max=np.inf, bw_max=np.inf):
    """Минимизируем сумму квадратов log(pred) - log(meas), то есть относительную ошибку,
    чтобы маленькие (0,1 мс) и большие (300 мс) точки весили одинаково.
    Параметры ищем в логарифмах: сначала сетка по (P, BW) с оптимальным t0, потом Нелдер-Мид.

    P и BW ограничены сверху паспортом GPU. Без этого задача вырождена: все слои
    растут как B * S^2, поэтому по (S, B) нельзя отличить «медленные вычисления» от
    «медленной памяти». На T4 без границ находится второй минимум с той же ошибкой,
    но с P = 13 TFLOP/s, что больше пика T4 (8,1 TFLOP/s), то есть физически невозможно."""
    ly = np.log(y)
    rows = eq.layer_table(S, B)
    F = np.stack([np.broadcast_to(r["flops"], y.shape) for r in rows])
    M = np.stack([np.broadcast_to(r["bytes"], y.shape) for r in rows])

    def clip(p):
        t0, P, BW = np.exp(p)
        return t0, min(P, p_max), min(BW, bw_max)

    def loss(p):
        t0, P, BW = clip(p)
        pred = t0 + np.maximum(F / P, M / BW).sum(0)
        return float(np.mean((np.log(pred) - ly) ** 2))

    best = None
    for lP in np.linspace(np.log(1e11), np.log(min(5e13, p_max)), 40):
        for lBW in np.linspace(np.log(1e10), np.log(min(2e12, bw_max)), 40):
            for lt0 in np.linspace(np.log(1e-6), np.log(1e-1), 30):
                v = loss((lt0, lP, lBW))
                if best is None or v < best[0]:
                    best = (v, np.array([lt0, lP, lBW]))
    p = nelder_mead(loss, best[1])
    t0, P, BW = clip(p)
    return {"t0": float(t0), "peak_flops": float(P), "bandwidth": float(BW)}


def nelder_mead(f, x0, step=0.3, iters=2000, tol=1e-12):
    n = len(x0)
    pts = [np.array(x0, float)] + [np.array(x0, float) + step * np.eye(n)[i] for i in range(n)]
    vals = [f(p) for p in pts]
    for _ in range(iters):
        order = np.argsort(vals)
        pts = [pts[i] for i in order]
        vals = [vals[i] for i in order]
        if abs(vals[-1] - vals[0]) < tol:
            break
        c = np.mean(pts[:-1], axis=0)
        xr = c + (c - pts[-1]); fr = f(xr)
        if fr < vals[0]:
            xe = c + 2 * (c - pts[-1]); fe = f(xe)
            pts[-1], vals[-1] = (xe, fe) if fe < fr else (xr, fr)
        elif fr < vals[-2]:
            pts[-1], vals[-1] = xr, fr
        else:
            xc = c + 0.5 * (pts[-1] - c); fc = f(xc)
            if fc < vals[-1]:
                pts[-1], vals[-1] = xc, fc
            else:
                pts = [pts[0]] + [pts[0] + 0.5 * (p - pts[0]) for p in pts[1:]]
                vals = [vals[0]] + [f(p) for p in pts[1:]]
    return pts[int(np.argmin(vals))]


# ------------------------------------------------------------------- energy
def fit_energy(S, B, y, theta):
    """E = p_static * Latency + e_flop * FLOPs + e_byte * Bytes, все коэффициенты >= 0.
    Линейная задача: взвешенные МНК по относительной ошибке, неотрицательность —
    перебором подмножеств (параметров всего три)."""
    X = np.stack([eq.latency(S, B, theta), eq.flops(S, B), eq.bytes_moved(S, B)], axis=1)
    Xw, yw = X / y[:, None], np.ones_like(y)
    scale = np.abs(Xw).max(0)
    best = None
    for mask in range(1, 8):
        idx = [j for j in range(3) if mask >> j & 1]
        c, *_ = np.linalg.lstsq(Xw[:, idx] / scale[idx], yw, rcond=None)
        c = c / scale[idx]
        if np.any(c < 0):
            continue
        coef = np.zeros(3)
        coef[idx] = c
        err = float(np.mean((Xw @ coef - 1) ** 2))
        if best is None or err < best[0]:
            best = (err, coef)
    p, ef, eb = best[1]
    return {"p_static": float(p), "e_flop": float(ef), "e_byte": float(eb)}


def memory_breakdown(S, B, meas, pred, runtime_bytes):
    """Где формула памяти точна и откуда остаток.
    Точная точка: |замер - формула| <= 1,1 МиБ (округление allocator-а и 1 МиБ рабочей области conv1).
    В остальных пик переезжает с maxpool на conv2 (5x5): живы a2 + a3 и рабочая область cuDNN.
    Из замера получаем её размер в байтах на B * S^2."""
    diff = meas - pred
    exact = np.abs(diff) <= 1.1 * 2**20
    static = eq.WEIGHT_BYTES + runtime_bytes + eq.alloc(eq.FP32 * B * 3 * S ** 2)
    a = eq.act_elems(S)
    conv2_live = eq.alloc(eq.FP32 * B * a["a2"]) + eq.alloc(eq.FP32 * B * a["a3"])
    ws = (meas - static - conv2_live)[~exact] / (B[~exact] * S[~exact] ** 2)
    return {"n_exact": int(exact.sum()), "n_total": int(len(meas)),
            "n_with_cudnn_workspace": int((~exact).sum()),
            "conv2_workspace_bytes_per_BS2_min": float(ws.min()) if len(ws) else None,
            "conv2_workspace_bytes_per_BS2_median": float(np.median(ws)) if len(ws) else None,
            "conv2_workspace_bytes_per_BS2_max": float(ws.max()) if len(ws) else None}


def metrics(pred, meas):
    rel = pred / meas - 1
    return {"mape_pct": float(100 * np.mean(np.abs(rel))), "max_abs_err_pct": float(100 * np.max(np.abs(rel))),
            "median_abs_err_pct": float(100 * np.median(np.abs(rel))), "n": int(len(rel))}


def main():
    d = load()
    ok = d["status"] == "ok"
    tr = ok & (d["is_validation"] == 0)
    va = ok & (d["is_validation"] == 1)
    S, B = d["S"], d["B"]
    runtime = json.load(open(RES / "runtime.json"))

    p_max, bw_max = gpu_bounds(runtime.get("gpu", ""))
    theta = fit_latency(S[tr], B[tr], d["latency_s"][tr], p_max, bw_max)
    has_e = np.isfinite(d["energy_j"][ok]).all()
    theta_e = fit_energy(S[tr], B[tr], d["energy_j"][tr], theta) if has_e else None

    pred_lat = eq.latency(S, B, theta)
    pred_mem = eq.memory(S, B, runtime_bytes=runtime["runtime_bytes"])
    pred_e = eq.energy(S, B, theta_e, theta) if has_e else np.full(len(S), np.nan)

    out = {
        "latency_theta": theta,
        "energy_theta": theta_e,
        "memory": {"runtime_bytes_measured": runtime["runtime_bytes"],
                   "note": "память не калибруется; runtime_bytes измерен напрямую (рабочая область cuBLAS)"},
        "split": {"train": "базовая сетка 7 x 9", "validation": "конфигурации со случайными S или B",
                  "n_train": int(tr.sum()), "n_validation": int(va.sum()), "n_oom": int((~ok).sum())},
        "latency_train": metrics(pred_lat[tr], d["latency_s"][tr]),
        "latency_validation": metrics(pred_lat[va], d["latency_s"][va]),
        "memory_all": metrics(pred_mem[ok], d["memory_bytes"][ok]),
        "memory_max_abs_err_mib": float(np.max(np.abs(pred_mem[ok] - d["memory_bytes"][ok])) / 2**20),
        "memory_exact": memory_breakdown(S[ok], B[ok], d["memory_bytes"][ok], pred_mem[ok], runtime["runtime_bytes"]),
        "latency_bounds": {"gpu": runtime.get("gpu", ""), "p_max_flops": p_max, "bw_max_bytes_per_s": bw_max},
    }
    if has_e:
        out["energy_train"] = metrics(pred_e[tr], d["energy_j"][tr])
        out["energy_validation"] = metrics(pred_e[va], d["energy_j"][va])
    json.dump(out, open(RES / "theta.json", "w"), indent=1, ensure_ascii=False)

    with open(RES / "predictions.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["S", "B", "is_validation", "status", "latency_s", "pred_latency_s", "memory_bytes",
                    "pred_memory_bytes", "energy_j", "pred_energy_j", "flops", "bytes_moved", "regime"])
        reg = eq.regime(S, B, theta)
        for i in range(len(S)):
            w.writerow([int(S[i]), int(B[i]), int(d["is_validation"][i]), d["status"][i], d["latency_s"][i],
                        pred_lat[i], d["memory_bytes"][i], pred_mem[i], d["energy_j"][i], pred_e[i],
                        eq.flops(S[i], B[i]), eq.bytes_moved(S[i], B[i]), ["launch", "memory", "compute"][int(reg[i])]])
    print(json.dumps(out, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile plots.py
"""Графики: измерения (точки) и предсказания формул (линии/поверхности) вместе.

Запуск после calibrate.py:  python plots.py  ->  results/figures/*.png
"""
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, ListedColormap

import equations as eq
from calibrate import load

RES = Path(__file__).resolve().parent / "results"
FIG = RES / "figures"

# Палитра: категориальные слоты (обучение / валидация / третья роль) и
# последовательная синяя шкала для упорядоченного S.
C_TRAIN, C_VAL, C_THIRD = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
SEQ = LinearSegmentedColormap.from_list("seq", ["#86b6ef", "#3987e5", "#256abf", "#104281", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#eb6834"])
MS = 7

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "lines.linewidth": 2,
})


def s_color(S, S_all):
    lo, hi = np.log(min(S_all)), np.log(max(S_all))
    return SEQ((np.log(S) - lo) / (hi - lo + 1e-12))


def save(fig, name):
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / name)
    plt.close(fig)
    print("saved", FIG / name)


def main():
    d = load()
    th = json.load(open(RES / "theta.json"))
    rt = json.load(open(RES / "runtime.json"))
    theta, theta_e = th["latency_theta"], th["energy_theta"]
    runtime = rt["runtime_bytes"]
    gpu = rt.get("gpu", "GPU")
    cap = rt.get("gpu_total_memory_bytes", np.nan)

    S, B = d["S"], d["B"]
    ok = d["status"] == "ok"
    oom = ~ok
    val = d["is_validation"] == 1
    tr = ok & ~val
    va = ok & val
    S_all = sorted(set(S.tolist()))
    base_S = [s for s in S_all if s not in rt.get("random_S", [])]
    Bc = np.unique(np.concatenate([np.geomspace(1, 256, 200), np.arange(1, 257)]))
    has_e = theta_e is not None and np.isfinite(d["energy_j"][ok]).all()

    p_lat = eq.latency(S, B, theta)
    p_mem = eq.memory(S, B, runtime_bytes=runtime)
    p_e = eq.energy(S, B, theta_e, theta) if has_e else np.full(len(S), np.nan)

    # ------------------------------------------------ 1. предсказание против измерения
    panels = [("Latency", d["latency_s"] * 1e3, p_lat * 1e3, "мс", "latency_train", "latency_validation"),
              ("Пиковая память", d["memory_bytes"] / 2**20, p_mem / 2**20, "МиБ", "memory_all", None)]
    if has_e:
        panels.append(("Энергия", d["energy_j"], p_e, "Дж", "energy_train", "energy_validation"))
    fig, axes = plt.subplots(1, len(panels), figsize=(5.2 * len(panels), 4.8))
    for ax, (title, meas, pred, unit, k_tr, k_va) in zip(np.atleast_1d(axes), panels):
        lo = np.nanmin(np.concatenate([meas[ok], pred[ok]])) * 0.7
        hi = np.nanmax(np.concatenate([meas[ok], pred[ok]])) * 1.4
        g = np.geomspace(lo, hi, 50)
        ax.fill_between(g, g * 0.9, g * 1.1, color=GRID, alpha=0.8, lw=0, label="±10%")
        ax.plot(g, g, color=MUTED, lw=1, label="идеал: предсказание = измерение")
        if k_va:
            ax.scatter(meas[tr], pred[tr], s=28, color=C_TRAIN, edgecolor="white", lw=0.6,
                       label=f"калибровка (базовая сетка), MAPE {th[k_tr]['mape_pct']:.1f}%")
            ax.scatter(meas[va], pred[va], s=28, color=C_VAL, edgecolor="white", lw=0.6, marker="D",
                       label=f"валидация (новые S или B), MAPE {th[k_va]['mape_pct']:.1f}%")
        else:
            ax.scatter(meas[ok], pred[ok], s=28, color=C_TRAIN, edgecolor="white", lw=0.6,
                       label=f"все точки, без калибровки, MAPE {th[k_tr]['mape_pct']:.2f}%")
        ax.set(xscale="log", yscale="log", xlim=(lo, hi), ylim=(lo, hi), title=title,
               xlabel=f"измерено, {unit}", ylabel=f"предсказано, {unit}")
        ax.legend(loc="upper left", fontsize=8)
    fig.suptitle(f"Предсказание формул против измерений на {gpu}", fontsize=12)
    save(fig, "01_pred_vs_measured.png")

    # ------------------------------------------------ 2. latency и throughput от B
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.8))
    for s in S_all:
        c = s_color(s, S_all)
        m = ok & (S == s)
        dash = "-" if s in base_S else "--"
        a1.plot(Bc, eq.latency(s, Bc, theta) * 1e3, dash, color=c, lw=1.6)
        a2.plot(Bc, Bc / eq.latency(s, Bc, theta), dash, color=c, lw=1.6)
        fc = c if s in base_S else "white"
        a1.scatter(B[m], d["latency_s"][m] * 1e3, s=22, facecolor=fc, edgecolor=c, zorder=3)
        a2.scatter(B[m], B[m] / d["latency_s"][m], s=22, facecolor=fc, edgecolor=c, zorder=3)
        if m.any():
            bl = B[m].max()
            a1.annotate(f"S={int(s)}", (bl, d["latency_s"][m & (B == bl)][0] * 1e3), xytext=(4, 0),
                        textcoords="offset points", fontsize=7, color=INK2, va="center")
    a1.axhline(theta["t0"] * 1e3, color=MUTED, lw=1, ls=":")
    a1.text(1.1, theta["t0"] * 1e3 * 1.15, f"t0 = {theta['t0'] * 1e3:.3f} мс (launch-bound пол)", fontsize=8, color=INK2)
    a1.set(xscale="log", yscale="log", xlabel="batch B", ylabel="latency, мс",
           title="Latency: точки — замеры, линии — формула")
    a2.set(xscale="log", yscale="log", xlabel="batch B", ylabel="картинок в секунду",
           title="Throughput: рост, пока GPU недогружена, потом плато")
    for a in (a1, a2):
        a.set_xticks([1, 2, 4, 8, 16, 32, 64, 128, 256], ["1", "2", "4", "8", "16", "32", "64", "128", "256"])
    a1.plot([], [], "-", color=SEQ(0.6), label="S базовой сетки (калибровка)")
    a1.plot([], [], "--", color=SEQ(0.6), label="случайные S (валидация)")
    a1.legend(loc="upper left", fontsize=8)
    save(fig, "02_latency_throughput_vs_batch.png")

    # ------------------------------------------------ 3. достигнутая производительность
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    F = eq.flops(S, B)
    for s in S_all:
        c = s_color(s, S_all)
        m = ok & (S == s)
        ax.scatter(F[m], F[m] / d["latency_s"][m] / 1e12, s=24, color=c, edgecolor="white", lw=0.5, zorder=3)
        ax.plot(eq.flops(s, Bc), eq.flops(s, Bc) / eq.latency(s, Bc, theta) / 1e12, color=c, lw=1.2, alpha=0.8)
    ax.axhline(theta["peak_flops"] / 1e12, color=C_VAL, lw=1.2, ls="--")
    ax.text(F[ok].min(), theta["peak_flops"] / 1e12 * 1.05, f"P = {theta['peak_flops'] / 1e12:.2f} TFLOP/s (подобрано)", color=INK2, fontsize=8)
    sm = plt.cm.ScalarMappable(cmap=SEQ, norm=matplotlib.colors.LogNorm(min(S_all), max(S_all)))
    fig.colorbar(sm, ax=ax, label="размер картинки S, пикс.")
    ax.set(xscale="log", yscale="log", xlabel="FLOPs одного прохода", ylabel="достигнуто, TFLOP/s",
           title="Мало работы — GPU простаивает (launch-bound), много — упираемся в P и BW")
    save(fig, "03_achieved_flops.png")

    # ------------------------------------------------ 4. из чего складывается время
    cfgs = [(32, 1), (64, 8), (128, 32), (224, 64), (384, 128), (512, 128)]
    cfgs = [c for c in cfgs if (ok & (S == c[0]) & (B == c[1])).any()]
    fig, ax = plt.subplots(figsize=(9, 4.6))
    for i, (s, b) in enumerate(cfgs):
        parts = {"launch": theta["t0"], "memory": 0.0, "compute": 0.0}
        for r in eq.layer_table(s, b):
            tc, tm = float(r["flops"]) / theta["peak_flops"], float(r["bytes"]) / theta["bandwidth"]
            parts["compute" if tc >= tm else "memory"] += max(tc, tm)
        tot = sum(parts.values())
        bottom = 0
        for k, col, lab in [("launch", MUTED, "постоянная задержка t0 (launch)"),
                            ("memory", C_VAL, "ядра, ограниченные памятью (ReLU, pool, 1x1)"),
                            ("compute", C_TRAIN, "ядра, ограниченные вычислениями")]:
            ax.bar(i, 100 * parts[k] / tot, bottom=bottom, color=col, width=0.6, edgecolor="white", lw=2,
                   label=lab if i == 0 else None)
            bottom += 100 * parts[k] / tot
        meas = d["latency_s"][ok & (S == s) & (B == b)][0]
        ax.text(i, 102, f"{tot * 1e3:.2f} мс\n(замер {meas * 1e3:.2f})", ha="center", fontsize=8, color=INK2)
    ax.set_xticks(range(len(cfgs)), [f"S={s}\nB={b}" for s, b in cfgs])
    ax.set(ylabel="% предсказанного времени", ylim=(0, 118),
           title="Смена режимов: launch-bound → memory-bound → compute-bound")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=8)
    ax.grid(axis="x", visible=False)
    save(fig, "04_time_breakdown.png")

    # ------------------------------------------------ 5. карта режимов
    Sg = np.linspace(32, 512, 120)[:, None]
    Bg = np.geomspace(1, 256, 120)[None, :]
    reg = eq.regime(Sg, Bg, theta)
    fig, ax = plt.subplots(figsize=(7.5, 5))
    cmap = ListedColormap(["#d6d5cf", "#f6c3ae", "#b7d3f6"])
    ax.pcolormesh(Bg.ravel(), Sg.ravel(), reg, cmap=cmap, vmin=-0.5, vmax=2.5, shading="auto")
    cs = ax.contour(Bg.ravel(), Sg.ravel(), eq.latency(Sg, Bg, theta) * 1e3, levels=[0.5, 1, 3, 10, 30, 100, 300],
                    colors=INK2, linewidths=0.7)
    ax.clabel(cs, fmt="%g мс", fontsize=7)
    ax.scatter(B[tr], S[tr], s=14, color=C_TRAIN, label="калибровка")
    ax.scatter(B[va], S[va], s=16, color=C_VAL, marker="D", label="валидация")
    if oom.any():
        ax.scatter(B[oom], S[oom], s=40, color="#d03b3b", marker="x", label="OOM")
    for y, lab, col in [(0, "launch-bound", "#d6d5cf"), (1, "memory-bound", "#f6c3ae"), (2, "compute-bound", "#b7d3f6")]:
        ax.scatter([], [], s=80, marker="s", color=col, label=lab)
    ax.set(xscale="log", xlabel="batch B", ylabel="размер картинки S, пикс.",
           title="Какая часть времени преобладает (по формуле), изолинии latency")
    ax.set_xticks([1, 2, 4, 8, 16, 32, 64, 128, 256], ["1", "2", "4", "8", "16", "32", "64", "128", "256"])
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
    save(fig, "05_regime_map.png")

    # ------------------------------------------------ 6. память
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.8))
    for s in S_all:
        c = s_color(s, S_all)
        m = ok & (S == s)
        a1.plot(Bc, eq.memory(s, Bc, runtime) / 2**20, color=c, lw=1.4)
        a1.scatter(B[m], d["memory_bytes"][m] / 2**20, s=22, color=c, edgecolor="white", lw=0.5, zorder=3)
    if np.isfinite(cap):
        a1.axhline(cap / 2**20, color="#d03b3b", lw=1.2, ls="--")
        a1.text(1.1, cap / 2**20 * 1.1, f"вся память GPU: {cap / 2**30:.1f} ГиБ", fontsize=8, color=INK2)
    if oom.any():
        a1.scatter(B[oom], p_mem[oom] / 2**20, marker="x", s=50, color="#d03b3b", zorder=4, label="OOM (точка — предсказание)")
        a1.legend(loc="lower right", fontsize=8)
    a1.set(xscale="log", yscale="log", xlabel="batch B", ylabel="пик памяти, МиБ",
           title="Пик памяти: точки — замер, линии — формула")
    resid = (d["memory_bytes"] - p_mem) / 2**20
    exact = ok & (np.abs(resid) <= 1.1)
    ws = ok & ~exact
    a2.axhline(0, color=MUTED, lw=1)
    a2.scatter(p_mem[exact] / 2**20, resid[exact], s=24, color=C_TRAIN, edgecolor="white", lw=0.5,
               label=f"формула точна (±1 МиБ): {exact.sum()} из {ok.sum()}")
    a2.scatter(p_mem[ws] / 2**20, resid[ws], s=24, color=C_VAL, edgecolor="white", lw=0.5, marker="D",
               label=f"cuDNN выбрал алгоритм conv2 с рабочей областью: {ws.sum()}")
    a2.legend(loc="upper left", fontsize=8)
    a2.set(xscale="log", xlabel="предсказанный пик, МиБ", ylabel="замер − формула, МиБ",
           title="Остаток = рабочая область cuDNN у conv2")
    save(fig, "06_memory.png")

    # ------------------------------------------------ 7. память по слоям
    try:
        import csv
        au = list(csv.DictReader(open(RES / "memory_audit.csv")))
        x = np.arange(len(au))
        fig, ax = plt.subplots(figsize=(9, 4.2))
        ax.bar(x - 0.2, [float(r["predicted_bytes"]) / 2**20 for r in au], 0.38, color=C_TRAIN, label="формула")
        ax.bar(x + 0.2, [float(r["measured_bytes"]) / 2**20 for r in au], 0.38, color=C_VAL, label="замер")
        ax.set_xticks(x, [r["layer"] for r in au])
        ax.set(ylabel="живые активации во время слоя, МиБ",
               title=f"Память по слоям (S={au[0]['S']}, B={au[0]['B']}): тензоры совпадают, у conv2 сверху рабочая область cuDNN")
        ax.legend()
        ax.grid(axis="x", visible=False)
        save(fig, "07_memory_by_layer.png")
    except FileNotFoundError:
        pass

    # ------------------------------------------------ 8. энергия
    if has_e:
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.8))
        for s in S_all:
            c = s_color(s, S_all)
            m = ok & (S == s)
            fc = c if s in base_S else "white"
            a1.plot(Bc, eq.energy(s, Bc, theta_e, theta) / Bc * 1e3, color=c, lw=1.4)
            a1.scatter(B[m], d["energy_j"][m] / B[m] * 1e3, s=22, facecolor=fc, edgecolor=c, zorder=3)
            a2.scatter(B[m], d["avg_power_w"][m], s=22, facecolor=fc, edgecolor=c, zorder=3)
        a1.set(xscale="log", yscale="log", xlabel="batch B", ylabel="энергия на картинку, мДж",
               title="Энергия на картинку: точки — замер, линии — формула")
        a2.axhline(theta_e["p_static"], color=MUTED, ls=":", lw=1)
        a2.text(1.1, theta_e["p_static"] * 1.02, f"p_static = {theta_e['p_static']:.1f} Вт", fontsize=8, color=INK2)
        a2.set(xscale="log", xlabel="batch B", ylabel="средняя мощность GPU, Вт",
               title="Средняя мощность во время замера энергии")
        for a in (a1, a2):
            a.set_xticks([1, 2, 4, 8, 16, 32, 64, 128, 256], ["1", "2", "4", "8", "16", "32", "64", "128", "256"])
        save(fig, "08_energy.png")

    # ------------------------------------------------ 9. ошибки по сетке
    Bs = sorted(set(B.tolist()))
    maps = [("latency", d["latency_s"], p_lat)] + ([("энергия", d["energy_j"], p_e)] if has_e else [])
    fig, axes = plt.subplots(1, len(maps), figsize=(7.2 * len(maps), 5.2))
    for ax, (name, meas, pred) in zip(np.atleast_1d(axes), maps):
        Z = np.full((len(S_all), len(Bs)), np.nan)
        for i in np.nonzero(ok)[0]:
            Z[S_all.index(S[i]), Bs.index(B[i])] = 100 * (pred[i] / meas[i] - 1)
        lim = max(10.0, np.nanpercentile(np.abs(Z), 95))
        im = ax.imshow(Z, cmap=DIV, norm=TwoSlopeNorm(0, -lim, lim), aspect="auto", origin="lower")
        for i in range(len(S_all)):
            for j in range(len(Bs)):
                if np.isfinite(Z[i, j]):
                    ax.text(j, i, f"{Z[i, j]:+.0f}", ha="center", va="center", fontsize=6.5, color=INK)
                elif any(oom & (S == S_all[i]) & (B == Bs[j])):
                    ax.text(j, i, "OOM", ha="center", va="center", fontsize=6, color="#d03b3b")
        rs, rb = set(rt.get("random_S", [])), set(rt.get("random_B", []))
        ax.set_xticks(range(len(Bs)), [f"{int(b)}*" if b in rb else str(int(b)) for b in Bs], fontsize=8)
        ax.set_yticks(range(len(S_all)), [f"{int(s)}*" if s in rs else str(int(s)) for s in S_all], fontsize=8)
        ax.set(xlabel="batch B (* — случайные, валидация)", ylabel="S (* — случайные, валидация)",
               title=f"Ошибка формулы {name}, % (предсказание / замер − 1)")
        ax.grid(False)
        fig.colorbar(im, ax=ax, label="%")
    save(fig, "09_error_heatmaps.png")

    # ------------------------------------------------ 10. частота и мощность: троттлинг
    if "sm_clock_mhz" in d and np.isfinite(d["sm_clock_mhz"][ok]).any():
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
        Fo = eq.flops(S, B)
        cols = [s_color(s, S_all) for s in S[ok]]
        a1.scatter(Fo[ok], d["sm_clock_mhz"][ok], s=22, c=cols, edgecolor="white", lw=0.5)
        a1.set(xscale="log", xlabel="FLOPs одного прохода", ylabel="частота SM, МГц",
               title="Под нагрузкой T4 сбрасывает частоту (лимит мощности 70 Вт)")
        a2.scatter(Fo[ok], d["avg_power_w"][ok], s=22, c=cols, edgecolor="white", lw=0.5)
        a2.axhline(70, color="#d03b3b", lw=1, ls="--")
        a2.text(Fo[ok].min(), 70.8, "TDP 70 Вт", fontsize=8, color=INK2)
        a2.set(xscale="log", xlabel="FLOPs одного прохода", ylabel="средняя мощность, Вт",
               title="Мощность упирается в TDP")
        save(fig, "10_clock_power.png")


if __name__ == "__main__":
    main()

## Проверка формул без GPU-замеров
FLOPs и память для нескольких точек, закрытые формы.

In [ ]:
!python models.py
!python equations.py

## Замеры (≈10–15 минут)

In [ ]:
!python measure.py

## Калибровка theta (только на базовой сетке) и проверка на валидации

In [ ]:
!python calibrate.py

## Графики

In [ ]:
!python plots.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f)
    display(Image(f))

## Скачать результаты
Архив `results.zip` нужно положить в папку `hw1/` репозитория (распаковать как `results/`).

In [ ]:
!zip -qr results.zip results
try:
    from google.colab import files
    files.download('results.zip')
except Exception:
    print('Не Colab: скачайте results.zip из панели файлов')